# ACLED protest motive extraction with Qwen3-32B

This notebook:

- loads an ACLED CSV;
- uses `notes` as the protest description;
- extracts an abstract protest motive with Qwen3-32B;
- stores the raw model response and supporting evidence;
- uses deterministic decoding;
- resolves and records the exact Hugging Face model commit;
- archives the prompt, environment, metadata, checksums, and outputs;
- optionally saves the exact quantized model checkpoint used for inference.

**Recommended runtime:** Colab GPU, ideally an A100 40 GB or larger.

Before running, choose **Runtime → Change runtime type → GPU**.

In [ ]:
# ============================================================
# 1. Install dependencies
# ============================================================

# Colab already provides pandas and other core data packages.
# Do NOT upgrade pandas: google-colab currently requires pandas==2.2.3.

!pip install -q -U \
    "transformers>=4.51.0" \
    "accelerate>=1.0.0" \
    "bitsandbytes>=0.45.0" \
    "huggingface_hub>=0.30.0" \
    tqdm


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.4/57.4 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 149.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 22.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 63.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 842.9/842.9 kB 60.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.2/80.2 kB 9.7 MB/s eta 0:00:00


In [ ]:
# ============================================================
# 2. Mount Google Drive
# ============================================================

from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
# ============================================================
# 3. Imports and reproducibility settings
# ============================================================

import os
import sys
import re
import json
import random
import hashlib
import shutil
import platform
import subprocess
from pathlib import Path

import numpy as np
import pandas as pd
import torch

SEED = 12345

os.environ["PYTHONHASHSEED"] = str(SEED)
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True

if torch.cuda.is_available():
    torch.backends.cuda.matmul.allow_tf32 = False
    torch.backends.cudnn.allow_tf32 = False

torch.use_deterministic_algorithms(True, warn_only=True)

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("CUDA version:", torch.version.cuda)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"GPU memory: {vram_gb:.1f} GB")

PyTorch: 2.11.0+cu128
CUDA available: True
CUDA version: 12.8
GPU: NVIDIA A100-SXM4-80GB
GPU memory: 79.3 GB


In [ ]:
# ============================================================
# 4. Upload the ACLED CSV
# ============================================================

from google.colab import files

uploaded = files.upload()

csv_files = [name for name in uploaded.keys() if name.lower().endswith(".csv")]

if len(csv_files) != 1:
    raise ValueError("Please upload exactly one CSV file.")

INPUT_CSV = csv_files[0]

print("Input file:", INPUT_CSV)

Saving ACLED Data_2026-09-21_event_date_from_2018-01-01_event_date_to_2023-01-01.csv to ACLED Data_2026-09-21_event_date_from_2018-01-01_event_date_to_2023-01-01.csv
Input file: ACLED Data_2026-09-21_event_date_from_2018-01-01_event_date_to_2023-01-01.csv


In [ ]:
# ============================================================
# 5. Model and coding specification
# ============================================================

MODEL_ID = "Qwen/Qwen3-32B"

TEXT_COLUMN = "notes"
ID_COLUMN = "event_id_cnty"

# Set to e.g. 20 for a test run; use None for the full dataset.
MAX_ROWS = None

MAX_NEW_TOKENS = 180

# Saving the exact 4-bit checkpoint can require ~20 GB or more
# of Google Drive space. Set False if the exact HF commit SHA,
# prompt, package versions and raw outputs are sufficient.
SAVE_EXACT_MODEL_CHECKPOINT = True

SYSTEM_PROMPT = r'''
You are a research assistant coding the motives of protest events
from ACLED event descriptions.

Your task is to determine WHY the protesters protested.

Use ONLY information contained in the event description.
Do not infer motives from the identity of the protesters,
their political party, location, nationality, or outside knowledge.

The central output is an ABSTRACT MOTIVE.

The abstract motive should:
- describe the central grievance, demand, or objective;
- be general enough to be comparable across protest events;
- normally contain between 2 and 10 words;
- not contain people's names, organization names, place names,
  dates, or other event-specific proper nouns;
- preserve substantively important distinctions.

Examples:

Description:
"Residents protested against rising utility tariffs."
Abstract motive:
"opposition to rising utility costs"

Description:
"Residents demanded that electricity supply be restored."
Abstract motive:
"demand for restoration of public utilities"

Description:
"Protesters demanded an impartial investigation into the death
of a local resident."
Abstract motive:
"demand for investigation and accountability"

Description:
"Workers demanded payment of several months of unpaid wages."
Abstract motive:
"demand for payment of wage arrears"

Description:
"Activists protested against the military invasion of Ukraine."
Abstract motive:
"opposition to war"

If several motives are explicitly present:
- identify the most central motive as abstract_motive;
- place additional distinct motives in secondary_motives.

If the motive cannot be determined from the description, use:
"unclear or unspecified motive"

Return ONLY valid JSON with exactly these fields:

{
  "abstract_motive": "short generalized motive",
  "specific_motive": "brief description of the concrete grievance or demand",
  "secondary_motives": [],
  "evidence": "short exact passage from the description supporting the coding",
  "uncertain": false
}

Do not add explanations outside the JSON.
'''.strip()

print(SYSTEM_PROMPT)

You are a research assistant coding the motives of protest events
from ACLED event descriptions.

Your task is to determine WHY the protesters protested.

Use ONLY information contained in the event description.
Do not infer motives from the identity of the protesters,
their political party, location, nationality, or outside knowledge.

The central output is an ABSTRACT MOTIVE.

The abstract motive should:
- describe the central grievance, demand, or objective;
- be general enough to be comparable across protest events;
- normally contain between 2 and 10 words;
- not contain people's names, organization names, place names,
  dates, or other event-specific proper nouns;
- preserve substantively important distinctions.

Examples:

Description:
"Residents protested against rising utility tariffs."
Abstract motive:
"opposition to rising utility costs"

Description:
"Residents demanded that electricity supply be restored."
Abstract motive:
"demand for restoration of public utilities"

Desc

In [ ]:
# ============================================================
# 6. Resolve the exact Hugging Face model commit
# ============================================================

from huggingface_hub import HfApi

hf_api = HfApi()
model_info = hf_api.model_info(MODEL_ID, revision="main")

MODEL_REVISION = model_info.sha

print("Model:", MODEL_ID)
print("Exact Hugging Face commit SHA:", MODEL_REVISION)

Model: Qwen/Qwen3-32B
Exact Hugging Face commit SHA: 9216db5781bf21249d130ec9da846c4624c16137


In [ ]:
# ============================================================
# 7. Create replication directories
# ============================================================

def sha256_file(path, chunk_size=1024 * 1024):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            chunk = f.read(chunk_size)
            if not chunk:
                break
            h.update(chunk)
    return h.hexdigest()

specification_string = json.dumps(
    {
        "model_id": MODEL_ID,
        "system_prompt": SYSTEM_PROMPT,
        "max_new_tokens": MAX_NEW_TOKENS,
        "thinking": False,
        "do_sample": False,
        "seed": SEED
    },
    sort_keys=True
)

RUN_FINGERPRINT = hashlib.sha256(
    specification_string.encode("utf-8")
).hexdigest()[:12]

PROJECT_ROOT = Path(
    f"/content/drive/MyDrive/ACLED_protest_motives_Qwen3_32B_{RUN_FINGERPRINT}"
)

MODEL_DIR = PROJECT_ROOT / "model"
RESULTS_DIR = PROJECT_ROOT / "results"
METADATA_DIR = PROJECT_ROOT / "metadata"
INPUT_DIR = PROJECT_ROOT / "input"

for directory in [
    PROJECT_ROOT,
    MODEL_DIR,
    RESULTS_DIR,
    METADATA_DIR,
    INPUT_DIR
]:
    directory.mkdir(parents=True, exist_ok=True)

print("Replication directory:")
print(PROJECT_ROOT)

Replication directory:
/content/drive/MyDrive/ACLED_protest_motives_Qwen3_32B_72571885580e


In [ ]:
# ============================================================
# 8. Archive and inspect input data
# ============================================================

ARCHIVED_INPUT = INPUT_DIR / Path(INPUT_CSV).name

if not ARCHIVED_INPUT.exists():
    shutil.copy2(INPUT_CSV, ARCHIVED_INPUT)

input_hash = sha256_file(ARCHIVED_INPUT)

with open(METADATA_DIR / "input_sha256.txt", "w") as f:
    f.write(f"{input_hash}  {ARCHIVED_INPUT.name}\n")

with open(METADATA_DIR / "coding_prompt.txt", "w", encoding="utf-8") as f:
    f.write(SYSTEM_PROMPT)

df = pd.read_csv(ARCHIVED_INPUT)

assert TEXT_COLUMN in df.columns, f"Missing column: {TEXT_COLUMN}"
assert ID_COLUMN in df.columns, f"Missing column: {ID_COLUMN}"

df["_row_id"] = range(len(df))

if MAX_ROWS is not None:
    df_work = df.iloc[:MAX_ROWS].copy()
else:
    df_work = df.copy()

print("Rows in source:", len(df))
print("Rows to process:", len(df_work))
print("Input SHA256:", input_hash)
print()
print(df.columns.tolist())

display(
    df_work[
        [c for c in [ID_COLUMN, "event_date", "location", TEXT_COLUMN]
         if c in df_work.columns]
    ].head(10)
)

Rows in source: 7704
Rows to process: 7704
Input SHA256: c0354218719d9ef6ad0220213cc688b97e9e9f4e4db6e9e92ef4e9b5998a81cf

['event_id_cnty', 'event_date', 'year', 'time_precision', 'disorder_type', 'event_type', 'sub_event_type', 'actor1', 'assoc_actor_1', 'inter1', 'actor2', 'assoc_actor_2', 'inter2', 'interaction', 'civilian_targeting', 'iso', 'region', 'country', 'admin1', 'admin2', 'admin3', 'location', 'latitude', 'longitude', 'geo_precision', 'source', 'source_scale', 'notes', 'fatalities', 'tags', 'timestamp', '_row_id']


,event_id_cnty,event_date,location,notes
0,RUS3,2018-01-04,Orenburg,"On 4 January 2018, four hunters gathered in Or..."
1,RUS4,2018-01-05,Kazan,"On 5 January 2018, local residents of Kazan wi..."
2,RUS5,2018-01-05,Moscow - Unspecified,"On 5 July 2018, a dozen of employees of Teatre..."
3,RUS8,2018-01-06,Makushino,"On 6 January 2018, a demonstration against the..."
4,RUS9,2018-01-06,St. Petersburg - Unspecified,"On 6 January 2018, protesters associated to th..."
5,RUS12,2018-01-10,Tyumen,"On Janruary 10th 2018, demstrators gathered in..."
6,RUS18,2018-01-13,Vladikavkaz,"On 13 January 2018, residents of Vladikavkaz m..."
7,RUS19,2018-01-14,Moscow - Unspecified,"On 14 January 2018, members of the movement Es..."
8,RUS20,2018-01-14,Vladikavkaz,"On 14 January 2018, a protest action was held ..."
9,RUS24,2018-01-15,Krasnodar,"On 15 January 2018, members of the movement Es..."


In [ ]:
# ============================================================
# 9. Record software and hardware environment
# ============================================================

with open(METADATA_DIR / "pip_freeze.txt", "w") as f:
    subprocess.run(
        [sys.executable, "-m", "pip", "freeze"],
        stdout=f,
        text=True
    )

runtime_info = [
    f"Python: {sys.version}",
    f"Platform: {platform.platform()}",
]

try:
    nvidia_output = subprocess.check_output(
        ["nvidia-smi"],
        text=True
    )
except Exception as e:
    nvidia_output = repr(e)

runtime_info.append("\n===== NVIDIA-SMI =====\n")
runtime_info.append(nvidia_output)

with open(
    METADATA_DIR / "runtime_environment.txt",
    "w",
    encoding="utf-8"
) as f:
    f.write("\n".join(runtime_info))

print(nvidia_output)

Mon Sep 21 00:02:33 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-80GB          Off |   00000000:00:05.0 Off |                    0 |
| N/A   32C    P0             49W /  400W |       6MiB /  81920MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [ ]:
# ============================================================
# 10. Load exact Qwen3-32B revision in 4-bit NF4
# ============================================================

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig
)

if torch.cuda.is_available() and torch.cuda.is_bf16_supported():
    COMPUTE_DTYPE = torch.bfloat16
else:
    COMPUTE_DTYPE = torch.float16

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=COMPUTE_DTYPE
)

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID
)

tokenizer.padding_side = "left"

if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=quantization_config,
    device_map="auto",
    low_cpu_mem_usage=True
)

model.eval()

print("Loaded:", MODEL_ID)
print("Revision:", MODEL_REVISION)
print("Compute dtype:", COMPUTE_DTYPE)

model.safetensors.index.json:   0%|          | 0.00/58.3k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 17 files:   0%|          | 0/17 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/707 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

Loaded: Qwen/Qwen3-32B


NameError: name 'MODEL_REVISION' is not defined

In [ ]:
# ============================================================
# 11. Save model/run metadata
# ============================================================

import transformers
import accelerate
import bitsandbytes
import huggingface_hub

metadata = {
    "model_id": MODEL_ID,
    "quantization": {
        "load_in_4bit": True,
        "quantization_type": "nf4",
        "double_quantization": True,
        "compute_dtype": str(COMPUTE_DTYPE)
    },
    "generation": {
        "enable_thinking": False,
        "do_sample": False,
        "max_new_tokens": MAX_NEW_TOKENS,
        "seed": SEED
    },
    "software": {
        "python": sys.version,
        "torch": torch.__version__,
        "transformers": transformers.__version__,
        "accelerate": accelerate.__version__,
        "bitsandbytes": bitsandbytes.__version__,
        "huggingface_hub": huggingface_hub.__version__
    },
    "data": {
        "input_filename": ARCHIVED_INPUT.name,
        "input_sha256": input_hash,
        "number_rows": len(df),
        "text_column": TEXT_COLUMN,
        "id_column": ID_COLUMN
    },
    "run_fingerprint": RUN_FINGERPRINT
}

if torch.cuda.is_available():
    metadata["hardware"] = {
        "gpu": torch.cuda.get_device_name(0),
        "cuda": torch.version.cuda,
        "total_gpu_memory_bytes":
            torch.cuda.get_device_properties(0).total_memory
    }

with open(
    METADATA_DIR / "model_and_run_metadata.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(metadata, f, indent=2)

print(json.dumps(metadata, indent=2))

{
  "model_id": "Qwen/Qwen3-32B",
  "quantization": {
    "load_in_4bit": true,
    "quantization_type": "nf4",
    "double_quantization": true,
    "compute_dtype": "torch.bfloat16"
  },
  "generation": {
    "enable_thinking": false,
    "do_sample": false,
    "max_new_tokens": 180,
    "seed": 12345
  },
  "software": {
    "python": "3.13.15 (main, Aug  6 2026, 11:06:22) [GCC 13.3.0]",
    "torch": "2.11.0+cu128",
    "transformers": "5.17.0",
    "accelerate": "1.15.0",
    "bitsandbytes": "0.50.2",
    "huggingface_hub": "1.32.0"
  },
  "data": {
    "input_filename": "ACLED Data_2026-09-21_event_date_from_2018-01-01_event_date_to_2023-01-01.csv",
    "input_sha256": "c0354218719d9ef6ad0220213cc688b97e9e9f4e4db6e9e92ef4e9b5998a81cf",
    "number_rows": 7704,
    "text_column": "notes",
    "id_column": "event_id_cnty"
  },
  "run_fingerprint": "72571885580e",
  "hardware": {
    "gpu": "NVIDIA A100-SXM4-80GB",
    "cuda": "12.8",
    "total_gpu_memory_bytes": 85094825984
  }
}


In [ ]:
# ============================================================
# 12. Optionally save the exact quantized checkpoint
# ============================================================

if SAVE_EXACT_MODEL_CHECKPOINT:

    marker = MODEL_DIR / "MODEL_ARCHIVE_COMPLETE.txt"

    if marker.exists():
        print("Model checkpoint already archived.")
    else:
        print("Saving exact quantized checkpoint to Google Drive...")

        model.save_pretrained(
            MODEL_DIR,
            safe_serialization=True,
            max_shard_size="4GB"
        )

        tokenizer.save_pretrained(MODEL_DIR)

        with open(marker, "w") as f:
            f.write(
                f"Model: {MODEL_ID}\n"
                f"Quantization: bitsandbytes NF4 4-bit\n"
                f"Run fingerprint: {RUN_FINGERPRINT}\n"
            )

        print("Model checkpoint saved.")
else:
    print("Checkpoint saving disabled.")

Model checkpoint already archived.


In [ ]:
# ============================================================
# 13. Inference and JSON parsing functions
# ============================================================

def make_messages(description):
    return [
        {
            "role": "system",
            "content": SYSTEM_PROMPT
        },
        {
            "role": "user",
            "content": (
                "Code the motive of the following protest event.\n\n"
                f"EVENT DESCRIPTION:\n{description}"
            )
        }
    ]


def extract_json(text):
    text = text.strip()

    text = re.sub(r"^```(?:json)?", "", text).strip()
    text = re.sub(r"```$", "", text).strip()

    try:
        return json.loads(text)
    except Exception:
        pass

    first = text.find("{")
    last = text.rfind("}")

    if first >= 0 and last > first:
        candidate = text[first:last + 1]
        try:
            return json.loads(candidate)
        except Exception:
            return None

    return None


@torch.inference_mode()
def code_one_description(description):

    if pd.isna(description) or str(description).strip() == "":
        return {
            "parsed": {
                "abstract_motive": "unclear or unspecified motive",
                "specific_motive": None,
                "secondary_motives": [],
                "evidence": None,
                "uncertain": True
            },
            "raw_response": "",
            "parse_ok": True
        }

    messages = make_messages(str(description))

    formatted_prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False
    )

    inputs = tokenizer(
        formatted_prompt,
        return_tensors="pt"
    )

    # Works with standard single-GPU Colab loading.
    # If the model is ever sharded across multiple GPUs, use
    # a dedicated inference engine such as vLLM instead.
    first_device = next(model.parameters()).device

    inputs = {
        k: v.to(first_device)
        for k, v in inputs.items()
    }

    input_length = inputs["input_ids"].shape[1]

    generated = model.generate(
        **inputs,
        do_sample=False,
        max_new_tokens=MAX_NEW_TOKENS,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id
    )

    new_tokens = generated[0, input_length:]

    raw_response = tokenizer.decode(
        new_tokens,
        skip_special_tokens=True
    ).strip()

    parsed = extract_json(raw_response)

    return {
        "parsed": parsed,
        "raw_response": raw_response,
        "parse_ok": parsed is not None
    }

NameError: name 'torch' is not defined

In [ ]:
# ============================================================
# 14. Test on five observations
# ============================================================

for i in range(min(5, len(df_work))):

    row = df_work.iloc[i]

    print("=" * 100)
    print("EVENT:", row[ID_COLUMN])
    print()
    print(row[TEXT_COLUMN])
    print()

    result = code_one_description(row[TEXT_COLUMN])

    print("PARSE OK:", result["parse_ok"])
    print()

    if result["parsed"] is not None:
        print(json.dumps(
            result["parsed"],
            indent=2,
            ensure_ascii=False
        ))
    else:
        print(result["raw_response"])

    print()

EVENT: RUS3

On 4 January 2018, four hunters gathered in Orenburg for a small demonstration in defense of proper hunting and dog raising regulations.

PARSE OK: True

{
  "abstract_motive": "support for proper hunting and animal regulations",
  "specific_motive": "advocacy for proper hunting and dog raising regulations",
  "secondary_motives": [],
  "evidence": "defense of proper hunting and dog raising regulations",
  "uncertain": false
}

EVENT: RUS4

On 5 January 2018, local residents of Kazan with their dogs/pets gathered in local park to protest a new legislature, according to which bear-baiting shall be prohibited in the future. Around 50 individuals participated. Around 30 000 individuals in Tatarstan hold a hunting license.

PARSE OK: True

{
  "abstract_motive": "opposition to prohibition of bear-baiting",
  "specific_motive": "protesters opposed a new law that would prohibit bear-baiting",
  "secondary_motives": [],
  "evidence": "local residents of Kazan with their dogs/pets

In [ ]:
# ============================================================
# 15. Process the whole dataset with restartable checkpointing
# ============================================================

from tqdm.auto import tqdm

RAW_JSONL = RESULTS_DIR / "raw_model_outputs.jsonl"

completed = {}

if RAW_JSONL.exists():
    with open(RAW_JSONL, "r", encoding="utf-8") as f:
        for line in f:
            try:
                record = json.loads(line)
                completed[int(record["_row_id"])] = record
            except Exception:
                pass

print("Already completed:", len(completed))

for _, row in tqdm(df_work.iterrows(), total=len(df_work)):

    row_id = int(row["_row_id"])

    if row_id in completed:
        continue

    description = row[TEXT_COLUMN]

    try:
        output = code_one_description(description)
        parsed = output["parsed"]

        if parsed is None:
            record = {
                "_row_id": row_id,
                ID_COLUMN: row[ID_COLUMN],
                "abstract_motive": None,
                "specific_motive": None,
                "secondary_motives": None,
                "evidence": None,
                "uncertain": True,
                "parse_ok": False,
                "raw_model_response": output["raw_response"],
                "error": None
            }
        else:
            record = {
                "_row_id": row_id,
                ID_COLUMN: row[ID_COLUMN],
                "abstract_motive": parsed.get("abstract_motive"),
                "specific_motive": parsed.get("specific_motive"),
                "secondary_motives": parsed.get("secondary_motives", []),
                "evidence": parsed.get("evidence"),
                "uncertain": parsed.get("uncertain"),
                "parse_ok": True,
                "raw_model_response": output["raw_response"],
                "error": None
            }

    except Exception as e:
        record = {
            "_row_id": row_id,
            ID_COLUMN: row[ID_COLUMN],
            "abstract_motive": None,
            "specific_motive": None,
            "secondary_motives": None,
            "evidence": None,
            "uncertain": True,
            "parse_ok": False,
            "raw_model_response": None,
            "error": repr(e)
        }

    with open(RAW_JSONL, "a", encoding="utf-8") as f:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")

    completed[row_id] = record

print("Finished.")
print("Total completed:", len(completed))

NameError: name 'RESULTS_DIR' is not defined

In [ ]:
# ============================================================
# 15.2 SELF-CONTAINED RESUME
# Run this cell only after a fresh Colab restart
# ============================================================

import sys
import subprocess

# ------------------------------------------------------------
# A. Install required packages
# ------------------------------------------------------------

subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q", "-U",
    "transformers>=4.51.0",
    "accelerate>=1.0.0",
    "bitsandbytes>=0.45.0",
    "huggingface_hub>=0.30.0",
    "tqdm"
])

# ------------------------------------------------------------
# B. Imports + mount Google Drive
# ------------------------------------------------------------

import os
import re
import gc
import json
import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch

from tqdm.auto import tqdm
from google.colab import drive, files
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig
)

drive.mount("/content/drive")

if not torch.cuda.is_available():
    raise RuntimeError(
        "No GPU detected. In Colab choose "
        "Runtime -> Change runtime type -> GPU."
    )

# ------------------------------------------------------------
# C. Use your existing project folder
# ------------------------------------------------------------

DRIVE_ROOT = Path("/content/drive/MyDrive")

PROJECT_ROOT = (
    DRIVE_ROOT
    / "ACLED_protest_motives_Qwen3_32B_72571885580e"
)

if not PROJECT_ROOT.exists():
    raise FileNotFoundError(
        f"Project folder not found:\n{PROJECT_ROOT}"
    )

MODEL_DIR = PROJECT_ROOT / "model"
RESULTS_DIR = PROJECT_ROOT / "results"
METADATA_DIR = PROJECT_ROOT / "metadata"
INPUT_DIR = PROJECT_ROOT / "input"

RAW_JSONL = RESULTS_DIR / "raw_model_outputs.jsonl"

print("Using project folder:")
print(PROJECT_ROOT)

print("\nCheckpoint:")
print(RAW_JSONL)

# ------------------------------------------------------------
# D. If checkpoint is missing in Drive, upload it
# ------------------------------------------------------------

if not RAW_JSONL.exists():

    print("\nraw_model_outputs.jsonl not found.")
    print("Please upload your existing checkpoint file.")

    uploaded = files.upload()

    if "raw_model_outputs.jsonl" not in uploaded:
        raise ValueError(
            "Please upload a file named raw_model_outputs.jsonl"
        )

    RAW_JSONL.parent.mkdir(
        parents=True,
        exist_ok=True
    )

    RAW_JSONL.write_bytes(
        Path("raw_model_outputs.jsonl").read_bytes()
    )

    print("\nCheckpoint restored to:")
    print(RAW_JSONL)

# ------------------------------------------------------------
# E. Recover settings from previous run
# ------------------------------------------------------------

metadata_file = (
    METADATA_DIR
    / "model_and_run_metadata.json"
)

prompt_file = (
    METADATA_DIR
    / "coding_prompt.txt"
)

if not metadata_file.exists():
    raise FileNotFoundError(
        f"Metadata file not found:\n{metadata_file}"
    )

if not prompt_file.exists():
    raise FileNotFoundError(
        f"Prompt file not found:\n{prompt_file}"
    )

with open(
    metadata_file,
    "r",
    encoding="utf-8"
) as f:
    metadata = json.load(f)

with open(
    prompt_file,
    "r",
    encoding="utf-8"
) as f:
    SYSTEM_PROMPT = f.read().strip()

MODEL_ID = metadata["model_id"]

# model_revision may not exist in older metadata
MODEL_REVISION = metadata.get(
    "model_revision",
    None
)

# Try to recover revision from archive marker if possible
archive_marker = (
    MODEL_DIR
    / "MODEL_ARCHIVE_COMPLETE.txt"
)

if (
    MODEL_REVISION is None
    and archive_marker.exists()
):

    marker_text = archive_marker.read_text(
        encoding="utf-8"
    )

    match = re.search(
        r"Revision:\s*(\S+)",
        marker_text
    )

    if match:
        MODEL_REVISION = match.group(1)

TEXT_COLUMN = metadata["data"]["text_column"]
ID_COLUMN = metadata["data"]["id_column"]

MAX_NEW_TOKENS = int(
    metadata["generation"]["max_new_tokens"]
)

SEED = int(
    metadata["generation"]["seed"]
)

input_filename = (
    metadata["data"]["input_filename"]
)

ARCHIVED_INPUT = (
    INPUT_DIR
    / input_filename
)

# fallback if filename differs but there is exactly one CSV
if not ARCHIVED_INPUT.exists():

    csv_files = list(
        INPUT_DIR.glob("*.csv")
    )

    if len(csv_files) == 1:
        ARCHIVED_INPUT = csv_files[0]
    else:
        raise FileNotFoundError(
            f"Could not identify archived input CSV in:\n"
            f"{INPUT_DIR}"
        )

print("\nModel:")
print(MODEL_ID)

if MODEL_REVISION is not None:
    print("Model revision:")
    print(MODEL_REVISION)
else:
    print(
        "Model revision not recorded. "
        "Archived Drive model will be used."
    )

print("\nArchived input:")
print(ARCHIVED_INPUT)

# ------------------------------------------------------------
# F. Reproducibility settings
# ------------------------------------------------------------

os.environ["PYTHONHASHSEED"] = str(SEED)
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"

random.seed(SEED)
np.random.seed(SEED)

torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True

torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False

torch.use_deterministic_algorithms(
    True,
    warn_only=True
)

print(
    "\nGPU:",
    torch.cuda.get_device_name(0)
)

# ------------------------------------------------------------
# G. Reload archived input data
# ------------------------------------------------------------

df = pd.read_csv(
    ARCHIVED_INPUT
)

assert TEXT_COLUMN in df.columns, (
    f"Missing text column: {TEXT_COLUMN}"
)

assert ID_COLUMN in df.columns, (
    f"Missing ID column: {ID_COLUMN}"
)

df["_row_id"] = range(
    len(df)
)

df_work = df.copy()

print(
    "\nTotal observations:",
    len(df_work)
)

# ------------------------------------------------------------
# H. Read existing checkpoint
# ------------------------------------------------------------

completed = {}
bad_checkpoint_lines = 0

with open(
    RAW_JSONL,
    "r",
    encoding="utf-8"
) as f:

    for line in f:

        if not line.strip():
            continue

        try:

            record = json.loads(
                line
            )

            completed[
                int(record["_row_id"])
            ] = record

        except Exception:

            bad_checkpoint_lines += 1

# ensure file ends with newline before appending
if RAW_JSONL.stat().st_size > 0:

    with open(
        RAW_JSONL,
        "rb+"
    ) as f:

        f.seek(
            -1,
            2
        )

        if f.read(1) != b"\n":

            f.seek(
                0,
                2
            )

            f.write(
                b"\n"
            )

first_missing = next(
    (
        i
        for i in range(len(df_work))
        if i not in completed
    ),
    None
)

print(
    "\nAlready completed:",
    len(completed)
)

if completed:

    print(
        "Highest completed _row_id:",
        max(completed)
    )

if first_missing is None:

    print(
        "All rows are already complete."
    )

else:

    print(
        "First unfinished _row_id:",
        first_missing
    )

    print(
        "Rows remaining:",
        len(df_work) - len(completed)
    )

if bad_checkpoint_lines:

    print(
        "Invalid/incomplete checkpoint lines ignored:",
        bad_checkpoint_lines
    )

# ------------------------------------------------------------
# I. Load tokenizer + model
# ------------------------------------------------------------

old_model = globals().pop(
    "model",
    None
)

if old_model is not None:

    del old_model

    gc.collect()

    torch.cuda.empty_cache()

COMPUTE_DTYPE = (
    torch.bfloat16
    if torch.cuda.is_bf16_supported()
    else torch.float16
)

if archive_marker.exists():

    print(
        "\nLoading archived model from Google Drive:"
    )

    print(
        MODEL_DIR
    )

    tokenizer = (
        AutoTokenizer.from_pretrained(
            MODEL_DIR,
            local_files_only=True
        )
    )

    model = (
        AutoModelForCausalLM.from_pretrained(
            MODEL_DIR,
            device_map="auto",
            low_cpu_mem_usage=True,
            local_files_only=True
        )
    )

else:

    print(
        "\nNo complete local model archive found."
    )

    if MODEL_REVISION is None:
        raise RuntimeError(
            "The archived model was not found and the exact "
            "Hugging Face revision is not recorded in the metadata."
        )

    print(
        "Loading exact Hugging Face revision:"
    )

    print(
        MODEL_REVISION
    )

    quantization_config = (
        BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_use_double_quant=True,
            bnb_4bit_compute_dtype=COMPUTE_DTYPE
        )
    )

    tokenizer = (
        AutoTokenizer.from_pretrained(
            MODEL_ID,
            revision=MODEL_REVISION
        )
    )

    model = (
        AutoModelForCausalLM.from_pretrained(
            MODEL_ID,
            revision=MODEL_REVISION,
            quantization_config=quantization_config,
            device_map="auto",
            low_cpu_mem_usage=True
        )
    )

tokenizer.padding_side = "left"

if tokenizer.pad_token_id is None:

    tokenizer.pad_token = (
        tokenizer.eos_token
    )

model.eval()

print(
    "\nModel loaded."
)

# ------------------------------------------------------------
# J. Recreate inference functions
# ------------------------------------------------------------

def make_messages(description):

    return [
        {
            "role": "system",
            "content": SYSTEM_PROMPT
        },
        {
            "role": "user",
            "content": (
                "Code the motive of the following protest event.\n\n"
                f"EVENT DESCRIPTION:\n{description}"
            )
        }
    ]


def extract_json(text):

    text = text.strip()

    text = re.sub(
        r"^```(?:json)?",
        "",
        text
    ).strip()

    text = re.sub(
        r"```$",
        "",
        text
    ).strip()

    try:

        return json.loads(
            text
        )

    except Exception:

        pass

    first = text.find("{")
    last = text.rfind("}")

    if first >= 0 and last > first:

        candidate = (
            text[
                first:last + 1
            ]
        )

        try:

            return json.loads(
                candidate
            )

        except Exception:

            return None

    return None


@torch.inference_mode()
def code_one_description(description):

    if (
        pd.isna(description)
        or str(description).strip() == ""
    ):

        return {
            "parsed": {
                "abstract_motive":
                    "unclear or unspecified motive",
                "specific_motive":
                    None,
                "secondary_motives":
                    [],
                "evidence":
                    None,
                "uncertain":
                    True
            },
            "raw_response": "",
            "parse_ok": True
        }

    messages = make_messages(
        str(description)
    )

    formatted_prompt = (
        tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
            enable_thinking=False
        )
    )

    inputs = tokenizer(
        formatted_prompt,
        return_tensors="pt"
    )

    first_device = (
        next(
            model.parameters()
        ).device
    )

    inputs = {
        k: v.to(first_device)
        for k, v in inputs.items()
    }

    input_length = (
        inputs["input_ids"].shape[1]
    )

    generated = (
        model.generate(
            **inputs,
            do_sample=False,
            max_new_tokens=MAX_NEW_TOKENS,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id
        )
    )

    new_tokens = (
        generated[
            0,
            input_length:
        ]
    )

    raw_response = (
        tokenizer.decode(
            new_tokens,
            skip_special_tokens=True
        ).strip()
    )

    parsed = extract_json(
        raw_response
    )

    return {
        "parsed": parsed,
        "raw_response": raw_response,
        "parse_ok": parsed is not None
    }

# ------------------------------------------------------------
# K. Resume processing
# ------------------------------------------------------------

if first_missing is not None:

    print(
        f"\nRESUMING FROM _row_id {first_missing}"
    )

    print(
        "Existing completed rows will be skipped.\n"
    )

    remaining_ids = [
        i
        for i in range(len(df_work))
        if i not in completed
    ]

    for row_id in tqdm(
        remaining_ids,
        total=len(remaining_ids)
    ):

        row = df_work.iloc[
            row_id
        ]

        description = (
            row[TEXT_COLUMN]
        )

        try:

            output = (
                code_one_description(
                    description
                )
            )

            parsed = (
                output["parsed"]
            )

            if parsed is None:

                record = {
                    "_row_id":
                        row_id,

                    ID_COLUMN:
                        row[ID_COLUMN],

                    "abstract_motive":
                        None,

                    "specific_motive":
                        None,

                    "secondary_motives":
                        None,

                    "evidence":
                        None,

                    "uncertain":
                        True,

                    "parse_ok":
                        False,

                    "raw_model_response":
                        output["raw_response"],

                    "error":
                        None
                }

            else:

                record = {
                    "_row_id":
                        row_id,

                    ID_COLUMN:
                        row[ID_COLUMN],

                    "abstract_motive":
                        parsed.get(
                            "abstract_motive"
                        ),

                    "specific_motive":
                        parsed.get(
                            "specific_motive"
                        ),

                    "secondary_motives":
                        parsed.get(
                            "secondary_motives",
                            []
                        ),

                    "evidence":
                        parsed.get(
                            "evidence"
                        ),

                    "uncertain":
                        parsed.get(
                            "uncertain"
                        ),

                    "parse_ok":
                        True,

                    "raw_model_response":
                        output[
                            "raw_response"
                        ],

                    "error":
                        None
                }

        except Exception as e:

            record = {
                "_row_id":
                    row_id,

                ID_COLUMN:
                    row[ID_COLUMN],

                "abstract_motive":
                    None,

                "specific_motive":
                    None,

                "secondary_motives":
                    None,

                "evidence":
                    None,

                "uncertain":
                    True,

                "parse_ok":
                    False,

                "raw_model_response":
                    None,

                "error":
                    repr(e)
            }

        # Save each result immediately
        with open(
            RAW_JSONL,
            "a",
            encoding="utf-8"
        ) as f:

            f.write(
                json.dumps(
                    record,
                    ensure_ascii=False
                )
                + "\n"
            )

            f.flush()

        completed[
            row_id
        ] = record

# ------------------------------------------------------------
# L. Final status
# ------------------------------------------------------------

print(
    "\nFinished."
)

print(
    "Total completed:",
    len(completed)
)

print(
    "Checkpoint:",
    RAW_JSONL
)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Using project folder:
/content/drive/MyDrive/ACLED_protest_motives_Qwen3_32B_72571885580e

Checkpoint:
/content/drive/MyDrive/ACLED_protest_motives_Qwen3_32B_72571885580e/results/raw_model_outputs.jsonl

Model:
Qwen/Qwen3-32B
Model revision not recorded. Archived Drive model will be used.

Archived input:
/content/drive/MyDrive/ACLED_protest_motives_Qwen3_32B_72571885580e/input/ACLED Data_2026-09-21_event_date_from_2018-01-01_event_date_to_2023-01-01.csv

GPU: NVIDIA A100-SXM4-80GB

Total observations: 7704

Already completed: 6489
Highest completed _row_id: 6488
First unfinished _row_id: 6489
Rows remaining: 1215

Loading archived model from Google Drive:
/content/drive/MyDrive/ACLED_protest_motives_Qwen3_32B_72571885580e/model


Loading weights:   0%|          | 0/707 [00:00<?, ?it/s]


Model loaded.

RESUMING FROM _row_id 6489
Existing completed rows will be skipped.



  0%|          | 0/1215 [00:00<?, ?it/s]


Finished.
Total completed: 7704
Checkpoint: /content/drive/MyDrive/ACLED_protest_motives_Qwen3_32B_72571885580e/results/raw_model_outputs.jsonl


In [ ]:
# ============================================================
# 16. Build and save final dataset
# ============================================================

records = []

with open(RAW_JSONL, "r", encoding="utf-8") as f:
    for line in f:
        records.append(json.loads(line))

motives = pd.DataFrame(records)

motives = (
    motives
    .drop_duplicates(subset="_row_id", keep="last")
    .sort_values("_row_id")
)

motives["secondary_motives_json"] = (
    motives["secondary_motives"]
    .apply(
        lambda x: json.dumps(x, ensure_ascii=False)
        if isinstance(x, list)
        else None
    )
)

motives_for_merge = motives.drop(
    columns=["secondary_motives"],
    errors="ignore"
)

final_df = df_work.merge(
    motives_for_merge,
    on=["_row_id", ID_COLUMN],
    how="left"
)

FINAL_CSV = RESULTS_DIR / "ACLED_protests_with_Qwen3_motives.csv"
FINAL_PARQUET = RESULTS_DIR / "ACLED_protests_with_Qwen3_motives.parquet"

final_df.to_csv(FINAL_CSV, index=False)
final_df.to_parquet(FINAL_PARQUET, index=False)

print("Saved:")
print(FINAL_CSV)
print(FINAL_PARQUET)

display(
    final_df[
        [
            ID_COLUMN,
            TEXT_COLUMN,
            "abstract_motive",
            "specific_motive",
            "secondary_motives_json",
            "evidence",
            "uncertain",
            "parse_ok"
        ]
    ].head(20)
)

Saved:
/content/drive/MyDrive/ACLED_protest_motives_Qwen3_32B_72571885580e/results/ACLED_protests_with_Qwen3_motives.csv
/content/drive/MyDrive/ACLED_protest_motives_Qwen3_32B_72571885580e/results/ACLED_protests_with_Qwen3_motives.parquet


,event_id_cnty,notes,abstract_motive,specific_motive,secondary_motives_json,evidence,uncertain,parse_ok
0,RUS3,"On 4 January 2018, four hunters gathered in Or...",support for proper hunting and animal regulations,advocacy for proper hunting and dog raising re...,[],defense of proper hunting and dog raising regu...,False,True
1,RUS4,"On 5 January 2018, local residents of Kazan wi...",opposition to prohibition of bear-baiting,protesters opposed a new law that would prohib...,[],local residents of Kazan with their dogs/pets ...,False,True
2,RUS5,"On 5 July 2018, a dozen of employees of Teatre...",support for arrested individual,protest in support of Ukrainian Oleg Sentsov w...,[],peaceful picket protest in support of the arre...,False,True
3,RUS8,"On 6 January 2018, a demonstration against the...",opposition to pension reform,protesters opposed the proposed changes to the...,[],a demonstration against the proposed pension r...,False,True
4,RUS9,"On 6 January 2018, protesters associated to th...",demand for release of political prisoners,demanding the release of activists Viktor Fili...,[],protesters ... demanding release of anarchist-...,False,True
5,RUS12,"On Janruary 10th 2018, demstrators gathered in...",opposition to financial mismanagement and fraud,protest over mismanagement and fraud in the 'T...,[],protest over the mismanagement and fraud in th...,False,True
6,RUS18,"On 13 January 2018, residents of Vladikavkaz m...",demand for investigation and accountability,protesters supported the family's claim that V...,[],His family claims that he was tortured and de ...,False,True
7,RUS19,"On 14 January 2018, members of the movement Es...",opposition to historical revisionism,protesters objected to the omission of a label...,[],They disagreed with a decision... not to put a...,False,True
8,RUS20,"On 14 January 2018, a protest action was held ...",demand for investigation and accountability,protesters demanded justice for the beating an...,[],a protest action was held in Vladikavkaz by th...,False,True
9,RUS24,"On 15 January 2018, members of the movement Es...",opposition to historical revisionism,protesters opposed the omission of a label ind...,[],They disagreed with a decision... not to put a...,False,True


In [ ]:
# ============================================================
# 17. Basic quality control
# ============================================================

n = len(final_df)

n_parse_failed = (~final_df["parse_ok"].fillna(False)).sum()
n_uncertain = final_df["uncertain"].fillna(True).astype(bool).sum()
n_missing = final_df["abstract_motive"].isna().sum()

print(f"Observations:       {n}")
print(f"Parse failures:     {n_parse_failed}")
print(f"Uncertain codings:  {n_uncertain}")
print(f"Missing motives:    {n_missing}")

print()
print("Most frequent abstract motives:")
print()

display(
    final_df["abstract_motive"]
    .value_counts(dropna=False)
    .head(30)
    .rename_axis("abstract_motive")
    .reset_index(name="N")
)

Observations:       1071
Parse failures:     0
Uncertain codings:  0
Missing motives:    0

Most frequent abstract motives:



,abstract_motive,N
0,opposition to environmental degradation,43
1,demand for restoration of public utilities,31
2,opposition to rising utility costs,25
3,opposition to environmental destruction,21
4,opposition to environmental pollution,19
5,opposition to animal euthanasia legislation,14
6,opposition to animal euthanasia,12
7,opposition to deforestation,12
8,opposition to war,11
9,support for political prisoners,9


In [ ]:
# ============================================================
# 18. Save checksums for key replication files
# ============================================================

important_files = [
    ARCHIVED_INPUT,
    FINAL_CSV,
    FINAL_PARQUET,
    RAW_JSONL,
    METADATA_DIR / "coding_prompt.txt",
    METADATA_DIR / "model_and_run_metadata.json",
    METADATA_DIR / "pip_freeze.txt"
]

checksum_records = []

for path in important_files:
    if Path(path).exists():
        checksum_records.append({
            "file": str(Path(path).relative_to(PROJECT_ROOT)),
            "sha256": sha256_file(path)
        })

checksum_df = pd.DataFrame(checksum_records)

checksum_df.to_csv(
    METADATA_DIR / "file_checksums.csv",
    index=False
)

display(checksum_df)

,file,sha256
0,input/ACLED Data_2026-09-20_event_date_from_20...,a795fcd65ccbc016956787f825e505d4ec30c5321234fd...
1,results/ACLED_protests_with_Qwen3_motives.csv,e900ea9cbeb113adc308ca89f0da78ec45ce0e2abcb139...
2,results/ACLED_protests_with_Qwen3_motives.parquet,37c538b84f4059809d47eb494fe057acd2dea6eb38f7a4...
3,results/raw_model_outputs.jsonl,bc5e03bedce439663bbb5328492f272cec3775f46b4569...
4,metadata/coding_prompt.txt,c5aa9555e0f45a51b10feb04ea0806360d0a7c7d8d2215...
5,metadata/model_and_run_metadata.json,b1dfb7bae2bb3e66dcc61df35ddc68c687bdbc2fd844e8...
6,metadata/pip_freeze.txt,8a5091373a3902385565a7a374af1992d39017053a4d6d...


In [ ]:
# ============================================================
# 19. Optional: hash archived model files
# ============================================================

if SAVE_EXACT_MODEL_CHECKPOINT:

    model_checksum_records = []

    for path in sorted(MODEL_DIR.rglob("*")):
        if path.is_file():
            print("Hashing:", path.name)

            model_checksum_records.append({
                "file": str(path.relative_to(PROJECT_ROOT)),
                "size_bytes": path.stat().st_size,
                "sha256": sha256_file(path)
            })

    model_checksum_df = pd.DataFrame(model_checksum_records)

    model_checksum_df.to_csv(
        METADATA_DIR / "model_file_checksums.csv",
        index=False
    )

    display(model_checksum_df)
else:
    print("No local model checkpoint was saved.")

Hashing: MODEL_ARCHIVE_COMPLETE.txt
Hashing: chat_template.jinja
Hashing: config.json
Hashing: generation_config.json
Hashing: model-00001-of-00005.safetensors
Hashing: model-00002-of-00005.safetensors
Hashing: model-00003-of-00005.safetensors
Hashing: model-00004-of-00005.safetensors
Hashing: model-00005-of-00005.safetensors
Hashing: model.safetensors.index.json
Hashing: tokenizer.json
Hashing: tokenizer_config.json


,file,size_bytes,sha256
0,model/MODEL_ARCHIVE_COMPLETE.txt,140,abf2c55d940d69ff656a507e49a5def8602e484e1b51ae...
1,model/chat_template.jinja,4168,a55ee1b1660128b7098723e0abcd92caa0788061051c62...
2,model/config.json,2692,d11e76992196510320b4791a49c4093cb2a282917f3e5e...
3,model/generation_config.json,214,893b0dccf83626cdfdc498e5b2a255a935b4c3c693350a...
4,model/model-00001-of-00005.safetensors,3999514228,d6a6924ba0438b28a0845539cc3b8fa94116d25cd24d1d...
5,model/model-00002-of-00005.safetensors,3959532398,991db210ef8128044cb330fa43a476bb44e4ade9d22603...
6,model/model-00003-of-00005.safetensors,3957450799,71c194056520c8e2cf8b556c22ab6714658d246c601970...
7,model/model-00004-of-00005.safetensors,4000044886,ddb8078391279fc4374093f7c51e54d9d6a9d1a645f8c5...
8,model/model-00005-of-00005.safetensors,3295392179,da36fdc85c30510dfb4dc73bae25a4d10761887b5c84a2...
9,model/model.safetensors.index.json,275361,9bac118ad676997897ea9f5a21850e97b905db2f6afaad...


In [ ]:
# ============================================================
# 20. Write replication README
# ============================================================

readme = f'''
ACLED PROTEST MOTIVE CODING — REPLICATION ARCHIVE
=================================================

MODEL
-----
Model: {MODEL_ID}
Exact Hugging Face revision:
{MODEL_REVISION}

QUANTIZATION
------------
4-bit NF4 via bitsandbytes
Double quantization: yes
Compute dtype: {COMPUTE_DTYPE}

INFERENCE
---------
Qwen thinking mode: disabled
Sampling: disabled
Decoding: greedy
Max new tokens: {MAX_NEW_TOKENS}
Seed: {SEED}

INPUT DATA
----------
Filename:
{ARCHIVED_INPUT.name}

SHA256:
{input_hash}

Number of source observations:
{len(df)}

Text field:
{TEXT_COLUMN}

Event identifier:
{ID_COLUMN}

OUTPUT VARIABLES
----------------
abstract_motive:
Generalized description of the main protest grievance or demand.

specific_motive:
Event-specific description of the grievance or demand.

secondary_motives_json:
Additional explicitly stated motives.

evidence:
Short exact passage from the source description supporting the coding.

uncertain:
Whether the description is insufficient or ambiguous.

parse_ok:
Whether the model response could be parsed as valid JSON.

raw_model_response:
Unmodified output produced by Qwen.

REPLICATION FILES
-----------------
input/
    Original ACLED input file.

model/
    Exact 4-bit checkpoint and tokenizer, if checkpoint archiving
    was enabled.

metadata/coding_prompt.txt
    Exact coding instructions.

metadata/model_and_run_metadata.json
    Model, quantization, generation, software, hardware and
    data settings.

metadata/pip_freeze.txt
    Complete package environment from the actual run.

metadata/runtime_environment.txt
    Runtime and GPU information.

metadata/file_checksums.csv
    SHA256 hashes of major files.

metadata/model_file_checksums.csv
    SHA256 hashes of model files, if archived.

results/raw_model_outputs.jsonl
    Observation-level raw model output.

results/ACLED_protests_with_Qwen3_motives.csv
    Final coded dataset.

results/ACLED_protests_with_Qwen3_motives.parquet
    Equivalent Parquet version.

RUN FINGERPRINT
---------------
{RUN_FINGERPRINT}
'''.strip()

with open(
    PROJECT_ROOT / "README_REPLICATION.txt",
    "w",
    encoding="utf-8"
) as f:
    f.write(readme)

print(readme)

ACLED PROTEST MOTIVE CODING — REPLICATION ARCHIVE

MODEL
-----
Model: Qwen/Qwen3-32B
Exact Hugging Face revision:
9216db5781bf21249d130ec9da846c4624c16137

QUANTIZATION
------------
4-bit NF4 via bitsandbytes
Double quantization: yes
Compute dtype: torch.bfloat16

INFERENCE
---------
Qwen thinking mode: disabled
Sampling: disabled
Decoding: greedy
Max new tokens: 180
Seed: 12345

INPUT DATA
----------
Filename:
ACLED Data_2026-09-20_event_date_from_2023-01-01_event_date_to_2026-09-20.csv

SHA256:
a795fcd65ccbc016956787f825e505d4ec30c5321234fd3219c8cf226e63fb42

Number of source observations:
1071

Text field:
notes

Event identifier:
event_id_cnty

OUTPUT VARIABLES
----------------
abstract_motive:
Generalized description of the main protest grievance or demand.

specific_motive:
Event-specific description of the grievance or demand.

secondary_motives_json:
Additional explicitly stated motives.

evidence:
Short exact passage from the source description supporting the coding.

uncertai

In [ ]:
# ============================================================
# 21. Show final project directory
# ============================================================

print("Replication archive:")
print(PROJECT_ROOT)

print("\nTop-level contents:")
for p in sorted(PROJECT_ROOT.iterdir()):
    print("-", p.name)

In [ ]:
# ============================================================
# FULL CHUNK:
# Upload enriched ACLED file, create state intervention outcome,
# construct protest-motive text, and generate Qwen embeddings
# ============================================================

# If needed, install sentence-transformers
!pip -q install sentence-transformers

import pandas as pd
import numpy as np

from google.colab import files
from sentence_transformers import SentenceTransformer


# ------------------------------------------------------------
# 1. Upload CSV from your computer
# ------------------------------------------------------------

uploaded = files.upload()

# Use the first uploaded file
filename = next(iter(uploaded))

df = pd.read_csv(filename)

print("Loaded file:", filename)
print("Dataset shape:", df.shape)


# ------------------------------------------------------------
# 2. Check required columns
# ------------------------------------------------------------

required_cols = [
    "interaction",
    "abstract_motive",
    "specific_motive"
]

missing_cols = [col for col in required_cols if col not in df.columns]

if missing_cols:
    raise ValueError(
        f"Missing required columns: {missing_cols}\n\n"
        f"Available columns are:\n{df.columns.tolist()}"
    )

print("\nRequired columns found.")


# ------------------------------------------------------------
# 3. Create state-intervention outcome
# ------------------------------------------------------------
# 1 = Russian state forces intervened against protesters
# 0 = no state-force intervention

df["state_intervention"] = (
    df["interaction"].eq("State forces-Protesters")
).astype(int)

print("\nState intervention counts:")
print(df["state_intervention"].value_counts())

print("\nState intervention proportions:")
print(df["state_intervention"].value_counts(normalize=True))


# ------------------------------------------------------------
# 4. Construct motive text
# ------------------------------------------------------------
# We use ONLY the Qwen-extracted protest motives.
# We do NOT use the original ACLED notes, because those may
# explicitly mention police arrests/intervention and create
# outcome leakage.

df["motive_text"] = (
    df["abstract_motive"]
      .fillna("")
      .astype(str)
      .str.strip()
    + ". "
    + df["specific_motive"]
      .fillna("")
      .astype(str)
      .str.strip()
).str.strip()


# ------------------------------------------------------------
# 5. Keep only observations with motive information
# ------------------------------------------------------------

valid_motive = (
    df["motive_text"]
      .str.replace(".", "", regex=False)
      .str.strip()
      .ne("")
)

df_model = df.loc[valid_motive].copy().reset_index(drop=True)

print("\nObservations with usable motive text:", len(df_model))

print("\nState intervention in modeling sample:")
print(df_model["state_intervention"].value_counts())

print("\nProportions:")
print(df_model["state_intervention"].value_counts(normalize=True))


# ------------------------------------------------------------
# 6. Inspect a few motive texts
# ------------------------------------------------------------

print("\nExamples of motive text:\n")

for i in range(min(5, len(df_model))):
    print(f"{i}: {df_model.loc[i, 'motive_text']}")
    print()


# ------------------------------------------------------------
# 7. Load Qwen embedding model
# ------------------------------------------------------------

embedding_model = SentenceTransformer(
    "Qwen/Qwen3-Embedding-0.6B"
)


# ------------------------------------------------------------
# 8. Generate embeddings
# ------------------------------------------------------------

motive_embeddings = embedding_model.encode(
    df_model["motive_text"].tolist(),
    batch_size=32,
    show_progress_bar=True,
    normalize_embeddings=True
)

motive_embeddings = np.asarray(motive_embeddings)


# ------------------------------------------------------------
# 9. Check dimensions
# ------------------------------------------------------------

print("\nEmbedding matrix shape:")
print(motive_embeddings.shape)

print("\nNumber of observations in df_model:")
print(len(df_model))

assert motive_embeddings.shape[0] == len(df_model)

print("\nEverything matches correctly.")


# ------------------------------------------------------------
# 10. Define X and y for classifier
# ------------------------------------------------------------

X = motive_embeddings
y = df_model["state_intervention"].values

print("\nX shape:", X.shape)
print("y shape:", y.shape)
print("Positive cases:", y.sum())
print("Negative cases:", len(y) - y.sum())

Saving ACLED_protests_with_Qwen3_motives.csv to ACLED_protests_with_Qwen3_motives.csv
Loaded file: ACLED_protests_with_Qwen3_motives.csv
Dataset shape: (1071, 40)

Required columns found.

State intervention counts:
state_intervention
0    990
1     81
Name: count, dtype: int64

State intervention proportions:
state_intervention
0    0.92437
1    0.07563
Name: proportion, dtype: float64

Observations with usable motive text: 1071

State intervention in modeling sample:
state_intervention
0    990
1     81
Name: count, dtype: int64

Proportions:
state_intervention
0    0.92437
1    0.07563
Name: proportion, dtype: float64

Examples of motive text:

0: demand for investigation and accountability. demanding an impartial investigation into the death of a young resident

1: opposition to arrest and espionage charges. protest against the arrest of the head of a pro-Russian media outlet accused of espionage by Latvian authorities

2: opposition to rising utility costs. protest against rising 

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/17.2k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

...

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

tokenizer.json: downloading bytes:           |  0.00B            

config.json:   0%|          | 0.00/313 [00:00<?, ?B/s]

Batches:   0%|          | 0/34 [00:00<?, ?it/s]


Embedding matrix shape:
(1071, 1024)

Number of observations in df_model:
1071

Everything matches correctly.

X shape: (1071, 1024)
y shape: (1071,)
Positive cases: 81
Negative cases: 990


In [ ]:
# ============================================================
# CLUSTER PROTEST MOTIVES BY COSINE SIMILARITY
# AND CALCULATE STATE-INTERVENTION SHARE BY CLUSTER
# ============================================================

import numpy as np
import pandas as pd

from sklearn.cluster import AgglomerativeClustering
from sklearn.metrics.pairwise import cosine_similarity


# ------------------------------------------------------------
# 1. Choose cosine-similarity threshold
# ------------------------------------------------------------
# 0.80 means that motives must be quite semantically similar
# to end up in the same cluster.
#
# Higher threshold -> more, smaller, more homogeneous clusters
# Lower threshold  -> fewer, larger, broader clusters

similarity_threshold = 0.80

# sklearn works with cosine DISTANCE:
# distance = 1 - cosine similarity

distance_threshold = 1 - similarity_threshold


# ------------------------------------------------------------
# 2. Hierarchical clustering
# ------------------------------------------------------------
# Complete linkage is useful here because it avoids clusters
# that are connected only through a chain of intermediate cases.

cluster_model = AgglomerativeClustering(
    n_clusters=None,
    metric="cosine",
    linkage="complete",
    distance_threshold=distance_threshold
)

cluster_labels = cluster_model.fit_predict(motive_embeddings)

df_model["motive_cluster"] = cluster_labels


print("Number of motive clusters:", df_model["motive_cluster"].nunique())


# ------------------------------------------------------------
# 3. Find representative motive for each cluster
# ------------------------------------------------------------
# We choose the observation that is most similar, on average,
# to all other motives in its cluster (the cluster medoid).

def representative_motive(cluster_id):

    idx = np.where(cluster_labels == cluster_id)[0]

    # singleton cluster
    if len(idx) == 1:
        return df_model.iloc[idx[0]]["motive_text"]

    emb = motive_embeddings[idx]

    similarities = cosine_similarity(emb)

    mean_similarity = similarities.mean(axis=1)

    representative_idx = idx[np.argmax(mean_similarity)]

    return df_model.iloc[representative_idx]["motive_text"]


cluster_ids = np.unique(cluster_labels)

representatives = {
    cluster_id: representative_motive(cluster_id)
    for cluster_id in cluster_ids
}


# ------------------------------------------------------------
# 4. Summarize clusters
# ------------------------------------------------------------

cluster_summary = (
    df_model
    .groupby("motive_cluster")
    .agg(
        n_protests=("state_intervention", "size"),
        n_interventions=("state_intervention", "sum"),
        intervention_share=("state_intervention", "mean")
    )
    .reset_index()
)


# Add representative motive
cluster_summary["representative_motive"] = (
    cluster_summary["motive_cluster"]
    .map(representatives)
)


# ------------------------------------------------------------
# 5. Sort by intervention share
# ------------------------------------------------------------

cluster_summary = (
    cluster_summary
    .sort_values(
        ["intervention_share", "n_protests"],
        ascending=[False, False]
    )
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# 6. Display all clusters
# ------------------------------------------------------------

pd.set_option("display.max_colwidth", 150)

display(
    cluster_summary[
        [
            "motive_cluster",
            "n_protests",
            "n_interventions",
            "intervention_share",
            "representative_motive"
        ]
    ]
)

Number of motive clusters: 587


,motive_cluster,n_protests,n_interventions,intervention_share,representative_motive
0,145,3,3,1.0,support for Palestine. protesters expressed solidarity with Palestine
1,119,2,2,1.0,opposition to construction works. protesters attempted to stop ongoing construction in the Bitsevsky Forest
2,132,1,1,1.0,opposition to local development changes. protesters opposed the readjustment of a local apartment building yard
3,153,1,1,1.0,demand for women's rights and opposition to domestic violence. protest for women's economic and reproductive rights and against domestic violence
4,157,1,1,1.0,demand for repatriation of conscripted soldiers. support for the 'A Path Home' movement demanding the government bring back soldiers mobilized to ...
...,...,...,...,...,...
582,582,1,0,0.0,opposition to construction on archeological site. protesters opposed the building of a church on an archeological site
583,583,1,0,0.0,opposition to animal treatment regulations. protesters opposed new laws banning feeding stray animals and keeping them outside shelters
584,584,1,0,0.0,support for political prisoners. activists staged pickets in support of political prisoners
585,585,1,0,0.0,demand for constitutional rights enforcement. demand that the government uphold the constitutional rights of Russian citizens
